In [0]:
%sql
use catalog delta_lake_study_demo

In [0]:
%sql
use schema delta_lake_demo

In [0]:
%sql
create table delta_lake_demo.demo (
    id int,
    name string,
    amount double
)using delta

In [0]:
%sql
insert into delta_lake_demo.demo values (1, 'alice', 100.0), (2, 'bob', 200.0), (3, 'charlie', 300.0);
select * from demo;

In [0]:
%sql
alter table delta_lake_demo.demo
set tblproperties(delta.enableChangeDataFeed = true)

In [0]:
%sql
describe history delta_lake_demo.demo

In [0]:
%sql
update  demo set amount = amount * 1.1 where id = 2

In [0]:
%sql
insert into demo values (4, 'david', 400.0)
    

In [0]:
%sql
delete from demo where id = 1

In [0]:
%sql
describe history delta_lake_demo.demo

In [0]:
%sql
select * from table_changes("demo",2)

In [0]:
%sql
delete from demo where id = 2

In [0]:
%sql
select * from table_changes("demo",2)

In [0]:
%sql
select * from table_changes("demo",2,4)

In [0]:
%sql
drop table if exists demo_deleted;
create table demo_deleted (
    id int,
    name string,
    amount double,
    _change_type string,
    _commit_version int,
    _commit_timestamp timestamp
)using delta

In [0]:
%sql
insert into demo_deleted 
select id,name,amount,_change_type,_commit_version,_commit_timestamp from table_changes("demo",2)
where _change_type = 'delete'

In [0]:
%sql
select * from demo_deleted

In [0]:
from pyspark.sql.functions import col

(spark.readStream.format("delta")
 .option("readChangeFeed","true")
 .option("startingVersion",2)
 .table("demo")
 .filter(col("_change_type") == "delete")
 .select("*")
 .writeStream
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/delta_lake_study_demo/delta_lake_demo/delta_sample_vol")
 .trigger(availableNow=True)
 .table("demo_deleted_stream")
 )

In [0]:
%sql
select * from demo_deleted_stream

In [0]:
%sql
delete from demo where id = 3

In [0]:
%sql
insert into demo_deleted 
select id,name,amount,_change_type,_commit_version,_commit_timestamp from table_changes("demo",2)
where _change_type = 'delete'

In [0]:
%sql
select * from demo_deleted

In [0]:
from pyspark.sql.functions import col

(spark.readStream.format("delta")
 .option("readChangeFeed","true")
 .option("startingVersion",2)
 .table("demo")
 .filter(col("_change_type") == "delete")
 .select("*")
 .writeStream
 .outputMode("append")
 .option("checkpointLocation", "/Volumes/delta_lake_study_demo/delta_lake_demo/delta_sample_vol")
 .trigger(availableNow=True)
 .table("demo_deleted_stream")
 )

In [0]:
%sql
select * from demo_deleted_stream